# Bing keyword demand analysis
Validation and intent filtering for the two Bing keyword-demand exports supplied on 2026-08-13.

In [1]:
from pathlib import Path
import os
import ast
import pandas as pd

source_dir = Path(os.environ.get('DATACOST_BING_EXPORT_DIR', '.')).expanduser()
airtime = pd.read_csv(source_dir / 'KeywordStats_8_13_2026 (2).csv')
data_terms = pd.read_csv(source_dir / 'KeywordStats_8_13_2026 (1).csv')
for frame in (airtime, data_terms):
    frame['trend_sum'] = frame['Trends'].map(lambda value: sum(ast.literal_eval(value)))
    frame['trend_reconciles'] = frame['trend_sum'] == frame['Impressions']
{'airtime_rows': len(airtime), 'airtime_impressions': int(airtime.Impressions.sum()), 'data_rows': len(data_terms), 'data_impressions': int(data_terms.Impressions.sum()), 'all_trends_reconcile': bool(airtime.trend_reconciles.all() and data_terms.trend_reconciles.all())}

{'airtime_rows': 17,
 'airtime_impressions': 2582,
 'data_rows': 145,
 'data_impressions': 52249,
 'all_trends_reconcile': True}

In [2]:
pattern = r'(?i)(?:mtn|vodacom|telkom|cell c|airtime|data deals|unlimited data|buy data|transfer data|data balance|data bundle|mobile data|prepaid data|night data|social data|whatsapp data|sim only|rain )'
relevant_data = data_terms[data_terms.Keyword.str.contains(pattern, regex=True, na=False)].copy()
{'relevant_rows': len(relevant_data), 'relevant_impressions': int(relevant_data.Impressions.sum()), 'share_pct': round(relevant_data.Impressions.sum() / data_terms.Impressions.sum() * 100, 1)}

{'relevant_rows': 25,
 'relevant_impressions': 16564,
 'share_pct': np.float64(31.7)}

In [3]:
clusters = pd.DataFrame([
    {'cluster': 'Operator data deals', 'impressions': 2463 + 1956 + 1553 + 1182},
    {'cluster': 'Operator and generic unlimited data', 'impressions': 783 + 601 + 520 + 376 + 231 + 186},
    {'cluster': 'Transfer airtime', 'impressions': 1083 + 247},
    {'cluster': 'Transfer data', 'impressions': 631 + 386},
    {'cluster': 'Buy data', 'impressions': 587 + 376 + 358},
]).sort_values('impressions', ascending=False)
clusters

,cluster,impressions
0,Operator data deals,7154
1,Operator and generic unlimited data,2697
2,Transfer airtime,1330
4,Buy data,1321
3,Transfer data,1017


## Guardrails
These are demand estimates, not DataCost clicks, rankings, or landing-page performance. Trend arrays reconcile to totals but have no date labels. Broad non-telecom meanings of data are excluded, and volumes must not be added to the Bing performance export.